# 머신러닝 기초 1장. 결정트리

사회약학 연구방법 노트 '머신러닝 기초' 1장을 따라 하는 노트북입니다. 0장과 같은 가상 청구자료와 분할로 결정트리가 자료를 나누는 방법을 작은 표로 확인하고, 나무의 크기와 과적합, 검증 곡선, 격자 탐색으로 조절값을 고른 뒤 결과를 표와 그림으로 정리합니다. 셀을 위에서부터 차례로 실행하세요. 셀 4–5는 0장의 준비 셀을 다시 실행하는 것이고, 셀 6은 이 장에서 새로 만드는 나무용 전처리 함수입니다. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 가. 나무가 자료를 나누는 방법

환자 10명으로 지니 불순도와 나누기 기준을 계산합니다.

### 셀 1. 지니 불순도와 나누기 전후의 감소량

In [ ]:
import numpy as np
import pandas as pd

toy = pd.DataFrame({
    "age":   [52, 61, 47, 78, 83, 69, 88, 58, 74, 81],
    "n_ed":  [0, 0, 1, 2, 0, 0, 3, 1, 2, 0],
    "admit": [0, 0, 0, 1, 0, 0, 1, 0, 0, 1]},
    index=list("ABCDEFGHIJ"))           # 0장 가 절의 환자 10명

def gini(y):
    """입원 비율 p에서 지니 불순도 1 - p² - (1 - p)²"""
    p = np.mean(y)
    return 1 - p**2 - (1 - p)**2

def split_gini(col, cut):
    """col <= cut과 col > cut으로 나눈 뒤 불순도의 가중 평균"""
    left = toy["admit"][toy[col] <= cut]
    right = toy["admit"][toy[col] > cut]
    return (len(left) * gini(left)
            + len(right) * gini(right)) / len(toy)

g0 = gini(toy["admit"])
print("before split:", round(g0, 3))
for col, cut in [("n_ed", 1.5), ("age", 76)]:
    g = split_gini(col, cut)
    print(f"{col} <= {cut}: after {g:.3f}, decrease {g0 - g:.3f}")

### 셀 2. 나이의 모든 기준을 시험하기

In [ ]:
ages = np.sort(toy["age"].unique())
cuts = (ages[:-1] + ages[1:]) / 2        # 이웃한 두 나이의 가운데
rows = [(cut, (toy["age"] <= cut).sum(), g0 - split_gini("age", cut))
        for cut in cuts]
pd.DataFrame(rows, columns=["cut", "n_left", "decrease"]).round(3)

### 셀 3. 사이킷런으로 같은 일 하기

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

stump = DecisionTreeClassifier(max_depth=1, random_state=2026)
stump.fit(toy[["age", "n_ed"]], toy["admit"])
print(export_text(stump, feature_names=["age", "n_ed"],
                  show_weights=True))
new = pd.DataFrame({"age": [80], "n_ed": [2]})    # 새 환자 K
print(stump.predict_proba(new))

**0장의 준비를 다시 합니다.** 자료, 분할, 교차검증 겹, 파이프라인을 만든 뒤 공통 자료로 작은 나무를 맞춥니다.

### 셀 4. 0장의 준비 (자료, 분할, 교차검증 겹)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold, cross_val_score)
from sklearn.metrics import roc_auc_score

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
df = pd.read_csv(BASE + "ml_claims.csv")
y = df["admit_2023"]
X = df.drop(columns=["id", "admit_2023", "cost_2023"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)
cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
print(X_train.shape, X_test.shape, y_train.sum(), y_test.sum())

### 셀 5. 0장의 make_pipe

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_pipe(model, num=num_cols, cat=cat_cols):
    """0장 셀 13과 같습니다 (로지스틱 회귀와 비교할 때 씀)."""
    num_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())])
    cat_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])
    prep = ColumnTransformer([("num", num_steps, num),
                              ("cat", cat_steps, cat)])
    return Pipeline([("prep", prep), ("model", model)])

### 셀 6. 나무용 파이프라인 make_tree_pipe

In [ ]:
def make_tree_pipe(model, num=num_cols, cat=cat_cols):
    """나무용: 결측 대체와 원-핫만 하고 표준화는 하지 않습니다."""
    cat_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])
    prep = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", cat_steps, cat)],
        verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

prep = make_tree_pipe(DecisionTreeClassifier()).named_steps["prep"]
names = prep.fit(X_train).get_feature_names_out()
print(len(names))
print(names[:3], names[-4:])

### 셀 7. 공통 자료로 깊이 3의 나무

In [ ]:
small = make_tree_pipe(
    DecisionTreeClassifier(max_depth=3, random_state=2026))
small.fit(X_train, y_train)
tree3 = small.named_steps["model"]
print(export_text(tree3, feature_names=list(names),
                  show_weights=True, decimals=1))

### 셀 8. 나무 그림 (plot_tree)

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(13, 5.5))
plot_tree(tree3, feature_names=list(names),
          class_names=["no", "admit"], filled=True, impurity=True,
          fontsize=8, ax=ax)
plt.tight_layout()

### 셀 9. 잎의 입원 비율이 곧 예측 확률

In [ ]:
Z_train = small.named_steps["prep"].transform(X_train)
leaf = tree3.apply(Z_train)                  # 사람마다 도착한 잎 번호
p_train = small.predict_proba(X_train)[:, 1]
by_leaf = pd.DataFrame({"leaf": leaf, "admit": y_train.to_numpy(),
                        "proba": p_train}).groupby("leaf").agg(
    n=("admit", "size"), admit_rate=("admit", "mean"),
    proba=("proba", "first"))
by_leaf.round(4)

## 나. 나무의 크기와 과적합

깊이를 늘려 가며 학습·교차검증·시험 AUC를 비교하고, class_weight와 가지치기를 봅니다.

### 셀 10. 깊이를 늘리면

In [ ]:
from sklearn.model_selection import cross_validate

rows = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, None]:
    m = make_tree_pipe(DecisionTreeClassifier(max_depth=d,
                                              random_state=2026))
    cv = cross_validate(m, X_train, y_train, cv=skf,
                        scoring="roc_auc", return_train_score=True)
    m.fit(X_train, y_train)
    p_te = m.predict_proba(X_test)[:, 1]    # 현상을 보려고만 계산
    rows.append({"max_depth": str(d),
                 "leaves": m.named_steps["model"].get_n_leaves(),
                 "train": cv["train_score"].mean(),
                 "cv": cv["test_score"].mean(),
                 "cv_sd": cv["test_score"].std(),
                 "test": roc_auc_score(y_test, p_te)})
depth_tab = pd.DataFrame(rows)
depth_tab.round(3)

### 셀 11. 깊이별 AUC 그림

In [ ]:
num = depth_tab.iloc[:-1]                    # None을 뺀 10줄
d = num["max_depth"].astype(int)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(d, num["train"], "o-", label="Training folds")
ax.plot(d, num["cv"], "o-", label="Cross-validation")
ax.plot(d, num["test"], "o--", color="gray", label="Test set")
ax.set_xlabel("max_depth")
ax.set_ylabel("AUC")
ax.set_title("Deeper trees fit the training data better")
ax.legend()
plt.tight_layout()

### 셀 12. class_weight를 바꾸면

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import recall_score

for cw in [None, "balanced"]:
    m = make_tree_pipe(DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=50, class_weight=cw,
        random_state=2026))
    auc = cross_val_score(m, X_train, y_train, cv=skf,
                          scoring="roc_auc").mean()
    p_oof = cross_val_predict(m, X_train, y_train, cv=skf,
                              method="predict_proba")[:, 1]
    pred = (p_oof >= 0.5).astype(int)
    print(f"{str(cw):8s}  CV AUC {auc:.3f}  mean prob "
          f"{p_oof.mean():.3f}  predicted 1: {pred.sum():4d}  "
          f"recall {recall_score(y_train, pred):.3f}")

### 셀 13. 비용 복잡도 가지치기 경로

In [ ]:
Zt = prep.transform(X_train)                # 대체·원-핫만 한 48열
path = DecisionTreeClassifier(random_state=2026)\
    .cost_complexity_pruning_path(Zt, y_train)
alphas = path.ccp_alphas
print(len(alphas), alphas[:3].round(6), alphas[-2:].round(5))

show = list(alphas[1::6]) + [alphas[-1]]     # 0 빼고 6개에 1개
n_leaves = [DecisionTreeClassifier(ccp_alpha=a, random_state=2026)
            .fit(Zt, y_train).get_n_leaves() for a in show]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True)
ax[0].step(alphas[1:], path.impurities[1:], where="post")
ax[0].set_ylabel("Total impurity of leaves")
ax[1].step(show, n_leaves, where="post")
ax[1].set_ylabel("Number of leaves")
for a in ax:
    a.set_xscale("log")
    a.set_xlabel("ccp_alpha (log scale)")
plt.tight_layout()

## 다. 조절값 하나씩 바꿔 보기

validation_curve로 max_depth와 min_samples_leaf를 하나씩 바꿉니다.

### 셀 14. 검증 곡선 1. max_depth

In [ ]:
from sklearn.model_selection import validation_curve

depths = list(range(1, 16))
tr, va = validation_curve(
    make_tree_pipe(DecisionTreeClassifier(random_state=2026)),
    X_train, y_train, param_name="model__max_depth",
    param_range=depths, cv=skf, scoring="roc_auc", n_jobs=-1)
print(tr.shape, va.shape)
print("best depth:", depths[va.mean(axis=1).argmax()],
      " CV AUC:", va.mean(axis=1).max().round(3))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for s, lab in [(tr, "Training folds"), (va, "Cross-validation")]:
    m, sd = s.mean(axis=1), s.std(axis=1)
    ax.plot(depths, m, "o-", label=lab)
    ax.fill_between(depths, m - sd, m + sd, alpha=0.2)
ax.set_xlabel("max_depth")
ax.set_ylabel("AUC (mean +/- SD of 5 folds)")
ax.set_title("Validation curve: max_depth")
ax.legend()
plt.tight_layout()

### 셀 15. 검증 곡선 2. min_samples_leaf

In [ ]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100, 200, 300, 400, 600, 800, 1200]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for d in [None, 4]:
    tr, va = validation_curve(
        make_tree_pipe(DecisionTreeClassifier(max_depth=d,
                                              random_state=2026)),
        X_train, y_train, param_name="model__min_samples_leaf",
        param_range=leaf_sizes, cv=skf, scoring="roc_auc", n_jobs=-1)
    m, sd = va.mean(axis=1), va.std(axis=1)
    best = leaf_sizes[m.argmax()]
    print(f"max_depth={d}: best min_samples_leaf {best}, "
          f"CV AUC {m.max():.3f}")
    ax.plot(leaf_sizes, m, "o-", label=f"CV, max_depth={d}")
    ax.fill_between(leaf_sizes, m - sd, m + sd, alpha=0.15)
    if d is None:
        ax.plot(leaf_sizes, tr.mean(axis=1), "o--", color="gray",
                label="Training folds, max_depth=None")
ax.set_xscale("log")
ax.set_xlabel("min_samples_leaf (log scale)")
ax.set_ylabel("AUC")
ax.set_title("Validation curve: min_samples_leaf")
ax.legend(fontsize=8)
plt.tight_layout()

## 라. 격자 탐색으로 튜닝하기

GridSearchCV로 두 조절값을 함께 고릅니다. 셀마다 수십 초가 걸릴 수 있습니다.

### 셀 16. 격자 탐색 1. 처음 정한 범위

In [ ]:
from sklearn.model_selection import GridSearchCV

grid1 = {"model__max_depth": [3, 4, 5, 6, 8, 10],
         "model__min_samples_leaf": [10, 20, 50, 100, 200]}
gs1 = GridSearchCV(
    make_tree_pipe(DecisionTreeClassifier(random_state=2026)),
    grid1, cv=skf, scoring="roc_auc", n_jobs=-1,
    return_train_score=True)
t0 = time.perf_counter()
gs1.fit(X_train, y_train)
print("seconds:", round(time.perf_counter() - t0, 1))
n_comb = len(gs1.cv_results_["params"])
print("combinations:", n_comb, " fits:", n_comb * skf.get_n_splits())
print("best:", gs1.best_params_, round(gs1.best_score_, 4))

### 셀 17. 격자 탐색 2. 범위를 넓혀 다시

In [ ]:
grid = {"model__max_depth": [4, 5, 6, 7, 8, 10, 12],
        "model__min_samples_leaf": [100, 150, 200, 300, 400, 600]}
gs = GridSearchCV(
    make_tree_pipe(DecisionTreeClassifier(random_state=2026)),
    grid, cv=skf, scoring="roc_auc", n_jobs=-1,
    return_train_score=True)
t0 = time.perf_counter()
gs.fit(X_train, y_train)
print("seconds:", round(time.perf_counter() - t0, 1))
print("fits:", len(gs.cv_results_["params"]) * 5)
print("best:", gs.best_params_, round(gs.best_score_, 4))

### 셀 18. 격자를 사전의 목록으로, 지표를 둘로

In [ ]:
grid_list = [
    {"model__max_depth": [5, 6, 7],
     "model__min_samples_leaf": [150, 300]},
    {"model__max_leaf_nodes": [8, 10, 12, 16],
     "model__min_samples_leaf": [150, 300]}]
gs_two = GridSearchCV(
    make_tree_pipe(DecisionTreeClassifier(random_state=2026)),
    grid_list, cv=skf, n_jobs=-1,
    scoring={"auc": "roc_auc", "logloss": "neg_log_loss"},
    refit="auc")
gs_two.fit(X_train, y_train)
r2 = pd.DataFrame(gs_two.cv_results_)
print("combinations:", len(r2), " best:", gs_two.best_params_)
show = r2[["param_model__max_depth", "param_model__max_leaf_nodes",
           "param_model__min_samples_leaf", "mean_test_auc",
           "rank_test_auc", "mean_test_logloss", "rank_test_logloss"]]
show.columns = ["depth", "max_leaves", "min_leaf", "auc", "auc_rank",
                "logloss", "ll_rank"]
show.sort_values("auc_rank").round(4)

## 마. 결과를 표와 그림으로 보기

탐색 결과 표와 열지도, 시험 자료 평가, 로지스틱 회귀와의 비교, 변수 중요도, 나무 그림, 학습 곡선, 모형 저장.

### 셀 19. cv_results_를 표로

In [ ]:
res = pd.DataFrame(gs.cv_results_)
table = (res[["rank_test_score", "param_model__max_depth",
              "param_model__min_samples_leaf", "mean_test_score",
              "std_test_score", "mean_train_score"]]
         .rename(columns={"rank_test_score": "rank",
                          "param_model__max_depth": "max_depth",
                          "param_model__min_samples_leaf": "min_leaf",
                          "mean_test_score": "cv_auc",
                          "std_test_score": "cv_sd",
                          "mean_train_score": "train_auc"})
         .sort_values("rank"))
table.head(8).round(4)

### 셀 20. 두 조절값의 열지도

In [ ]:
heat = res.pivot(index="param_model__min_samples_leaf",
                 columns="param_model__max_depth",
                 values="mean_test_score")
fig, ax = plt.subplots(figsize=(7, 4.2))
im = ax.imshow(heat.to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns)
ax.set_yticks(range(heat.shape[0]), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iloc[i, j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                fontsize=8, color="white" if v > 0.745 else "black")
ax.set_xlabel("max_depth")
ax.set_ylabel("min_samples_leaf")
ax.set_title("Mean CV AUC")
fig.colorbar(im, ax=ax)
plt.tight_layout()

### 셀 21. 최종 나무를 시험 자료로 한 번 평가

In [ ]:
def boot_auc(y_true, p, n_boot=1000, seed=2026):
    """시험 자료를 n_boot번 복원추출해 AUC를 다시 구한 값들"""
    rng = np.random.default_rng(seed)
    yt, p = np.asarray(y_true), np.asarray(p)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        if yt[i].min() == yt[i].max():      # 한 범주만 뽑히면 건너뜀
            continue
        aucs.append(roc_auc_score(yt[i], p[i]))
    return np.array(aucs)

final = gs.best_estimator_              # 학습 자료 전체로 다시 맞춤
p_tree = final.predict_proba(X_test)[:, 1]
b_tree = boot_auc(y_test, p_tree)
print("leaves:", final.named_steps["model"].get_n_leaves())
print("test AUC:", round(roc_auc_score(y_test, p_tree), 3),
      " 95% CI:", np.percentile(b_tree, [2.5, 97.5]).round(3))

### 셀 22. 0장 로지스틱 회귀와 비교

In [ ]:
from sklearn.linear_model import LogisticRegression

# 0장 라 절에서 고른 C = 0.1
lr = make_pipe(LogisticRegression(C=0.1, max_iter=1000))
cv_lr = cross_val_score(lr, X_train, y_train, cv=skf,
                        scoring="roc_auc")
lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_test)[:, 1]
b_lr = boot_auc(y_test, p_lr)           # 같은 seed, 같은 복원추출

i_best = gs.best_index_
comp = pd.DataFrame({
    "cv_auc": [res.loc[i_best, "mean_test_score"], cv_lr.mean()],
    "cv_sd": [res.loc[i_best, "std_test_score"], cv_lr.std()],
    "test_auc": [roc_auc_score(y_test, p_tree),
                 roc_auc_score(y_test, p_lr)],
    "ci_low": [np.percentile(b_tree, 2.5), np.percentile(b_lr, 2.5)],
    "ci_high": [np.percentile(b_tree, 97.5),
                np.percentile(b_lr, 97.5)]},
    index=["decision tree", "logistic (C=0.1)"])
print("difference 95% CI:",
      np.percentile(b_tree - b_lr, [2.5, 97.5]).round(3))
comp.round(3)

### 셀 23. 두 모형의 ROC 곡선

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(4.8, 4.4))
for p, lab in [(p_tree, "Decision tree"), (p_lr, "Logistic (C=0.1)")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, label=f"{lab}, AUC "
            f"{roc_auc_score(y_test, p):.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("1 - specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC curves in the test set")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()

### 셀 24. 불순도 기반 변수 중요도

In [ ]:
deep = make_tree_pipe(DecisionTreeClassifier(random_state=2026))
deep.fit(X_train, y_train)              # 깊이 제한 없는 나무
imp_final = pd.Series(final.named_steps["model"].feature_importances_,
                      index=names).sort_values(ascending=False)
imp_deep = pd.Series(deep.named_steps["model"].feature_importances_,
                     index=names).sort_values(ascending=False)
print("nonzero in final tree:", (imp_final > 0).sum())
null4 = ["dyslip", "oa", "n_prescribers", "region_city"]
print("no-effect features, rank in deep tree:",
      [list(imp_deep.index).index(f) + 1 for f in null4])

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, imp, t in [(ax[0], imp_final, "Tuned tree (11 leaves)"),
                  (ax[1], imp_deep, "Unrestricted tree")]:
    top = imp[imp > 0].head(12)[::-1]   # 0보다 큰 것만
    a.barh(top.index, top.to_numpy())
    a.set_xlabel("Impurity-based importance")
    a.set_title(t)
plt.tight_layout()

### 셀 25. 최종 나무 그림

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(final.named_steps["model"], feature_names=list(names),
          class_names=["no", "admit"], filled=True, proportion=True,
          impurity=False, precision=3, fontsize=7, ax=ax)
plt.tight_layout()

### 셀 26. 학습 곡선 (learning_curve)

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr, va = learning_curve(
    make_tree_pipe(DecisionTreeClassifier(        # 셀 17에서 고른 조절값
        max_depth=6, min_samples_leaf=300, random_state=2026)),
    X_train, y_train, train_sizes=[0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
    cv=skf, scoring="roc_auc", n_jobs=-1)
print("patients:", sizes)
print("train:", tr.mean(axis=1).round(3))
print("CV:   ", va.mean(axis=1).round(3))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for s, lab in [(tr, "Training folds"), (va, "Cross-validation")]:
    m, sd = s.mean(axis=1), s.std(axis=1)
    ax.plot(sizes, m, "o-", label=lab)
    ax.fill_between(sizes, m - sd, m + sd, alpha=0.2)
ax.set_xlabel("Number of patients used for training")
ax.set_ylabel("AUC (mean +/- SD of 5 folds)")
ax.set_title("Learning curve: max_depth=6, min_samples_leaf=300")
ax.legend()
plt.tight_layout()

### 셀 27. 모형 저장과 불러오기 (joblib)

In [ ]:
import joblib

joblib.dump(final, "tree_admit_2023.joblib")       # 파일로 저장
loaded = joblib.load("tree_admit_2023.joblib")     # 다시 불러오기
p_again = loaded.predict_proba(X_test)[:, 1]
print("same predictions:", np.array_equal(p_again, p_tree))
print(loaded.named_steps["model"].get_params()["min_samples_leaf"])

## 과제 정답

**과제 1.** criterion을 "entropy"로 바꿔 셀 17과 같은 격자로 다시 튜닝하고 지니로 고른 결과와 비교합니다.

### 셀 28. 과제 1 정답. 엔트로피로 다시 튜닝

In [ ]:
gs_ent = GridSearchCV(
    make_tree_pipe(DecisionTreeClassifier(criterion="entropy",
                                          random_state=2026)),
    grid, cv=skf, scoring="roc_auc", n_jobs=-1)
gs_ent.fit(X_train, y_train)
print("entropy best:", gs_ent.best_params_,
      round(gs_ent.best_score_, 4))
print("gini best   :", gs.best_params_, round(gs.best_score_, 4))
print("leaves:", gs_ent.best_estimator_.named_steps["model"]
      .get_n_leaves())

**과제 2.** ccp_alpha만 바꾼 나무(깊이와 잎 크기 제한 없음)의 교차검증 AUC를 구해 깊이와 잎 크기로 제한한 나무와 비교합니다.

### 셀 29. 과제 2 정답. 가지치기한 나무와 크기를 제한한 나무

In [ ]:
alpha_grid = [0, 1e-4, 2e-4, 3e-4, 4e-4, 5e-4, 7e-4, 1e-3, 1.5e-3]
tr, va = validation_curve(
    make_tree_pipe(DecisionTreeClassifier(random_state=2026)),
    X_train, y_train, param_name="model__ccp_alpha",
    param_range=alpha_grid, cv=skf, scoring="roc_auc", n_jobs=-1)
for a, m in zip(alpha_grid, va.mean(axis=1)):
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=2026)
    t.fit(prep.transform(X_train), y_train)
    print(f"ccp_alpha {a:<7}  leaves {t.get_n_leaves():4d}"
          f"  CV AUC {m:.3f}")
print("depth/leaf tuned tree: leaves 11, CV AUC",
      round(gs.best_score_, 3))

**과제 3.** 고른 조절값의 나무를 class_weight 없이와 "balanced"로 맞춰 교차검증 예측 확률의 혼동행렬을 임계값 0.5와 입원 비율(0.041)에서 비교합니다.

### 셀 30. 과제 3 정답. class_weight와 임계값

In [ ]:
from sklearn.metrics import confusion_matrix

best = dict(max_depth=6, min_samples_leaf=300, random_state=2026)
rate = y_train.mean()                        # 학습 자료의 입원 비율
for cw, thr in [(None, 0.5), ("balanced", 0.5), (None, rate)]:
    m = make_tree_pipe(
        DecisionTreeClassifier(class_weight=cw, **best))
    p = cross_val_predict(m, X_train, y_train, cv=skf,
                          method="predict_proba")[:, 1]
    print(f"class_weight={cw}, threshold {thr:.3f}")
    print(confusion_matrix(y_train, (p >= thr).astype(int)))